In [1]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from langchain_community.document_loaders import (
    Docx2txtLoader,
    PyPDFLoader,
    TextLoader,
)

from langchain_core.documents import Document

from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)

from langchain_qdrant import QdrantVectorStore

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
)

from qdrant_client import QdrantClient


/var/folders/8n/sd4x57xj1rg948ljm67gsxw80000gn/T/ipykernel_20735/866064222.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [2]:
_ = load_dotenv(find_dotenv())

openai_api_key = os.environ["OPENAI_API_KEY"]

qdrant_api_key = os.environ["QDRANT_API_KEY"]

qdrant_url = os.environ["QDRANT_URL"]

qdrant_collection = os.getenv(
    "QDRANT_COLLECTION",
    "case-intelligence",
)

print("OpenAI key loaded:", bool(openai_api_key))
print("Qdrant key loaded:", bool(qdrant_api_key))
print("Qdrant URL:", qdrant_url)
print("Collection:", qdrant_collection)


OpenAI key loaded: True
Qdrant key loaded: True
Qdrant URL: https://79280cc3-a5d5-4068-904d-f2f06e4014a5.us-east-1-1.aws.cloud.qdrant.io:6333
Collection: case-intelligence


In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chat_model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

print("Models initialized")


Models initialized


In [4]:
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
)

collections = qdrant_client.get_collections()
print("collections")
print(collections)

print("Connected to Qdrant")

for collection in collections.collections:
    print("-", collection.name)


collections
CollectionsResponse({'collections': [{'name': 'case-intelligence'}]})
Connected to Qdrant
- case-intelligence


In [5]:
data_dir = Path("data")

documents_dir = data_dir / "documents"
transcripts_dir = data_dir / "transcripts"

print("Documents:", documents_dir)
print("Transcripts:", transcripts_dir)


Documents: data/documents
Transcripts: data/transcripts


In [6]:
supported_extensions = {
    ".pdf",
}

print(supported_extensions)


{'.pdf'}


In [7]:
def load_file(
    path: Path,
    source_type: str,
) -> list[Document]:

    extension = path.suffix.lower()

    print("Loading:", path)
    if extension == ".pdf":
        loader = PyPDFLoader(str(path))

    documents = loader.load()
    print("documents\n", documents)

    for document in documents:
        document.metadata["source"] = path.name

        document.metadata["source_type"] = source_type

        document.metadata["source_path"] = str(path)

        if "page" in document.metadata:
            document.metadata["page_number"] = document.metadata["page"] + 1

    print(
        "Loaded units:",
        len(documents),
    )

    return documents


In [8]:
def load_directory(
    directory: Path,
    source_type: str,
) -> list[Document]:

    documents: list[Document] = []

    for path in sorted(directory.rglob("*")):
        if not path.is_file():
            continue

        if path.suffix.lower() not in supported_extensions:
            continue

        loaded_documents = load_file(
            path=path,
            source_type=source_type,
        )

        documents.extend(loaded_documents)

    return documents


In [9]:
reference_documents = load_directory(
    directory=documents_dir,
    source_type="reference",
)

print(
    "Reference document units:",
    len(reference_documents),
)


Loading: data/documents/2022 Colorado Community Corrections Standards copy.pdf


Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)


documents

[Document({'id': None,
  'metadata': {'producer': 'macOS Version 14.6.1 (Build 23G93) Quartz PDFContext, AppendMode 1.1',
   'creator': 'Microsoft® Word 2016',
   'creationdate': "D:20221004191316Z00'00'",
   'subject': 'Standards for Colorado community corrections',
   'author': 'Office of Community Corrections (OCC)',
   'title': '2022 Colorado Community Corrections Standards',
   'moddate': "D:20250513223956Z00'00'",
   'keywords': 'community corrections, standards, client supervision, evidence-based practice',
   'source': 'data/documents/2022 Colorado Community Corrections Standards copy.pdf',
   'total_pages': 63,
   'page': 0,
   'page_label': '1'},
  'page_content': 'Colorado Community \nCorrections Standards \n \n \nState of Colorado  \nDepartment of Public Safety  \nDivision of Criminal Justice  \nOffice of Community Corrections  \n \n700 Kipling Street, Suite 1000  \nDenver, Colorado 80215  \ndcj.colorado.gov/dcj-offices/office-of-community-corrections',
  'type':

In [10]:
transcript_documents = load_directory(
    directory=transcripts_dir,
    source_type="transcript",
)

print(
    "Transcript document units:",
    len(transcript_documents),
)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)


Loading: data/transcripts/nathan-04-14.pdf
documents

[Document({'id': None,
  'metadata': {'producer': 'macOS Version 14.6.1 (Build 23G93) Quartz PDFContext',
   'creator': 'PyPDF',
   'creationdate': "D:20250612042647Z00'00'",
   'moddate': "D:20250612042647Z00'00'",
   'source': 'data/transcripts/nathan-04-14.pdf',
   'total_pages': 7,
   'page': 0,
   'page_label': '1'},
  'page_content': "I'm going to start the recording. \n241 Elms Court Circle still? \nYes, sir. \nPhone number is the same? \nYes, sir. \nObviously, smart plumbing still? \nYes, sir. \nDrug screen was negative on the 26th. \nHow's your ankle monitor doing? \nGood. \nNo issues with it? \nNo. \nKeeping it charged, all that good stuff? \nYes, sir. \nSweet. \nStill no medications? \nNo. \nAll right. \nFees. \nYou got $3.95, it looks like. \nokay i'll get that caught up i got house taxes right now so if it gets a little high i \nmean \nyeah if i can drag it out can everybody work with me yeah i got two thousand \ndollar

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)


documents

[Document({'id': None,
  'metadata': {'producer': 'macOS Version 14.6.1 (Build 23G93) Quartz PDFContext',
   'creator': 'PyPDF',
   'creationdate': "D:20250612042608Z00'00'",
   'moddate': "D:20250612042608Z00'00'",
   'source': 'data/transcripts/nathan-05-19.pdf',
   'total_pages': 8,
   'page': 0,
   'page_label': '1'},
  'page_content': "start \nall right here we go \nis your address 241 still yep phone number the 474 yeah smart plumbing \nyeah uh how's weekly drug \nscreens going good the weekly like pbt yeah yeah i gotta do it after i'm done \nwith you but yeah i'm \ndoing good good man uh drug screen was negative on the 14th yeah i don't \ndo drugs there you go \nuh ankle monitor doing good man no issues yeah they called me the other day \nlike an hour before \ni go to work and told me i needed to charge it i already charged it for the day \nso i had to charge \nit again but other than that why did they say why they said it was still in the \norange but i did my \nhour

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)


documents

[Document({'id': None,
  'metadata': {'producer': 'macOS Version 14.6.1 (Build 23G93) Quartz PDFContext',
   'creator': 'PyPDF',
   'creationdate': "D:20250612042505Z00'00'",
   'moddate': "D:20250612042505Z00'00'",
   'source': 'data/transcripts/nathan-06-02.pdf',
   'total_pages': 6,
   'page': 0,
   'page_label': '1'},
  'page_content': "There we go. \nSo I've got address at the 241 Elms Court. \nYes, ma'am. \nAnd then phone is 317-474-5726. \nYes, ma'am. \nAnd smart plumbing for your employer. \nAll right. \nAll right. Last drug screen was the first and it was negative. \nYesterday. \nyeah all right and how's the ankle monitor doing still good yes ma'am okay \nthank you \nall right no medication right and it looks like these are at a thousand and \ntwenty \none thousand and twenty yep okay i'll pay half of that tomorrow when i get \npaid okay i'll get \nthat caught up you need to put a note in there i've ran it up pretty high but i'll \nget it taken \ncare of that's all 

In [11]:
all_documents = reference_documents + transcript_documents

print(
    "Total document units:",
    len(all_documents),
)


Total document units: 107


In [12]:
for document in all_documents[:5]:
    print("=" * 80)

    # print("Metadata:")
    # print(document.metadata)

    # print()

    print("Content:")
    print(document.page_content[:100])

    # print()


Content:
Colorado Community 
Corrections Standards 
 
 
State of Colorado  
Department of Public Safety  
Div
Content:
2022 Colorado Community Corrections Standards  
Published: October 2022  1 | Page 
 
Table of Conten
Content:
2022 Colorado Community Corrections Standards  
Published: October 2022  2 | Page 
 
EF-090 Access t
Content:
2022 Colorado Community Corrections Standards  
Published: October 2022  3 | Page 
 
SD-020: Staff E
Content:
2022 Colorado Community Corrections Standards  
Published: October 2022  4 | Page 
 
CD-120: Level S


In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)


In [14]:
chunks = text_splitter.split_documents(all_documents)

print(
    "Documents before splitting:",
    len(all_documents),
)

print(
    "Chunks after splitting:",
    len(chunks),
)


Documents before splitting: 107
Chunks after splitting: 273


In [15]:
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_index"] = index

In [16]:
for chunk in chunks[:3]:
    print("=" * 80)

    print("Metadata:")
    print(chunk.metadata)

    print()

    print("Content:")
    print(chunk.page_content[:500])


Metadata:
{'producer': 'macOS Version 14.6.1 (Build 23G93) Quartz PDFContext, AppendMode 1.1',
 'creator': 'Microsoft® Word 2016',
 'creationdate': "D:20221004191316Z00'00'",
 'subject': 'Standards for Colorado community corrections',
 'author': 'Office of Community Corrections (OCC)',
 'title': '2022 Colorado Community Corrections Standards',
 'moddate': "D:20250513223956Z00'00'",
 'keywords': 'community corrections, standards, client supervision, evidence-based practice',
 'source': '2022 Colorado Community Corrections Standards copy.pdf',
 'total_pages': 63,
 'page': 0,
 'page_label': '1',
 'source_type': 'reference',
 'source_path': 'data/documents/2022 Colorado Community Corrections Standards copy.pdf',
 'page_number': 1,
 'chunk_index': 0}

Content:
Colorado Community 
Corrections Standards 
 
 
State of Colorado  
Department of Public Safety  
Division of Criminal Justice  
Office of Community Corrections  
 
700 Kipling Street, Suite 1000  
Denver, Colorado 80215  
dcj.colorado

In [17]:
if qdrant_client.collection_exists(qdrant_collection):
    qdrant_client.delete_collection(qdrant_collection)

    print(
        "Deleted old collection:",
        qdrant_collection,
    )
else:
    print("Collection does not exist yet")


Deleted old collection: case-intelligence


In [19]:
vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url=qdrant_url,
    api_key=qdrant_api_key,
    collection_name=qdrant_collection,
    timeout=60,
    batch_size=16,
)

print("Uploaded chunks to Qdrant")


Uploaded chunks to Qdrant


In [20]:
collections = qdrant_client.get_collections()

for collection in collections.collections:
    print(collection.name)


case-intelligence


In [21]:
question = "What are some important things Robert talks about?"

retrieved_docs = vector_store.similarity_search(
    question,
    k=6,
)

print(
    "Retrieved:",
    len(retrieved_docs),
)


Retrieved: 6


In [22]:
context_parts = []

for index, doc in enumerate(
    retrieved_docs,
    start=1,
):
    source = doc.metadata.get(
        "source",
        "Unknown",
    )

    page = doc.metadata.get("page_number")

    source_name = source

    if page:
        source_name += f" - Page {page}"

    context_parts.append(
        f"""
[SOURCE {index}]
Source: {source_name}

{doc.page_content}
""".strip()
    )


context = "\n\n".join(context_parts)

print(context[:4000])


[SOURCE 1]
Source: 8 Principles of Effective Intervention.pdf - Page 1

8 Principles of Effective Intervention 
 
 
Research supports several principles for effective offender interventions. NIC highlights 8 principles in its 
"Evidence-Based Policy and Practice" initiative. They are listed below in developmental sequence. 
Resources for implementing program interventions are also listed below.  
1. Assess Actuarial Risk/Needs - Assessing offenders' risk and needs (focusing on 
dynamic and static risk factors and criminogenic needs) at the individual and 
aggregate levels is essential for implementating the principles of best practice. 
2. Enhance Intrinsic Motivation - Research strongly suggests that "motivational 
interviewing" techniques, rather than persuasion tactics, effectively enhance 
motivation for initiating and maintaining behavior changes 
3. Target Interventions 
1. Risk Principle - Prioritize supervision and treatment resources for higher risk 
offenders. 
2. Need Princi

In [ ]:
system_prompt = """
You are a case intelligence assistant.

Answer the user's question using only
the retrieved evidence.

Rules:

- Do not use outside knowledge.
- Do not invent facts.
- If there is not enough evidence,
  clearly say so.
- You may combine information from
  multiple sources.
- Cite your claims using
  [SOURCE 1], [SOURCE 2], etc.
"""


In [ ]:
response = chat_model.invoke(
    [
        (
            "system",
            system_prompt,
        ),
        (
            "human",
            f"""
            Question:
            {question}
            
            Evidence:
            {context}
            """,
        ),
    ]
)


In [ ]:
print(response.content)

In [ ]:
for index, doc in enumerate(
    retrieved_docs,
    start=1,
):
    print("=" * 80)

    print(f"[SOURCE {index}]")

    print(
        "File:",
        doc.metadata.get("source"),
    )

    print(
        "Type:",
        doc.metadata.get("source_type"),
    )

    print(
        "Page:",
        doc.metadata.get("page_number"),
    )

    print()

    print(doc.page_content[:800])
